In [ ]:
%pip install langchain-community langchain langchain-astradb

In [ ]:
%pip install -qU langchain-google-genai

In [ ]:
%pip install unstructured "unstructured[pdf]" "unstructured[pptx]"

In [ ]:
!sudo apt-get update

In [ ]:
!sudo apt-get install poppler-utils

In [ ]:
!sudo apt-get install libleptonica-dev tesseract-ocr libtesseract-dev python3-pil tesseract-ocr-eng tesseract-ocr-script-latn

In [ ]:
%pip install unstructured-pytesseract

In [ ]:
%pip install tesseract-ocr

In [ ]:
%pip install datasets pypdf

In [ ]:
%pip install -U langchain-text-splitters

In [ ]:
%pip install langchain-core

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders.directory import DirectoryLoader

In [ ]:
loader = DirectoryLoader(path="/content/data")

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=64)

In [ ]:
docs = loader.load_and_split(text_splitter=splitter)

In [ ]:
len(docs)

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI

In [ ]:
from google.colab import userdata
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

In [ ]:
import os

os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY

In [ ]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001"
)

In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite"
)

In [ ]:
from langchain_astradb import AstraDBVectorStore

In [ ]:
vector_store = AstraDBVectorStore(
    embedding=embeddings,
    collection_name='multidocs',
    api_endpoint=ASTRADB_API_ENDPOINT,
    token=ASTRADB_TOKEN,
    namespace='default_keyspace'
)

In [ ]:
inserted_ids = vector_store.add_documents(docs)

In [ ]:
prompt = """
You are an AI philosopher drawing insights from the docs of "rag" and "genai".
Craft thoughtful answers based on this roadmap, mixing and matching existing paths.
Your responses should be concise and strictly related to the provided context.

ROADMAP CONTEXT:
{context}

QUESTION: {question}

YOUR ANSWER:"""

In [ ]:
prompt_template = ChatPromptTemplate.from_template(prompt)

In [ ]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

In [ ]:
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt_template
    | llm
    | StrOutputParser()
)

In [ ]:
chain.invoke("What is the importance of RAG in AI?")